In [38]:
# %% Cell 2 - Setup, Corpus, Retrieval, Telemetry, LLM Wrapper
import os, re, json, time, random, textwrap
import numpy as np
import pandas as pd
import chromadb
from google import genai
from google.genai import types
from sentence_transformers import SentenceTransformer
from google.colab import userdata

client = genai.Client(api_key=userdata.get("cns_project_key"))

MODEL = "gemini-3.5-flash-lite"
TEMP      = 0.1
MAX_TOK   = 800
SIM_MIN   = 0.30
ROLES     = ["General User", "Firmware Developer", "Network Administrator"]

N_CALLS, N_EMPTY = 0, 0
def gen(prompt):
    """One API call with strict universal pacing and retries."""
    global N_CALLS, N_EMPTY
    time.sleep(4.5)
    for attempt in range(6):
        try:
            r = client.models.generate_content(
                model=MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction="You are an expert IoT security assistant.",
                    temperature=TEMP,
                    max_output_tokens=MAX_TOK,
                )
            )
            N_CALLS += 1
            text = (r.text or "").strip()
            if not text:
                N_EMPTY += 1
            return text
        except Exception as e:
            print("  [retry]", attempt, str(e)[:100])
            time.sleep(15 * (attempt + 1))
    N_EMPTY += 1
    return ""

def tele(cpu, conns, anomaly): return dict(cpu=cpu, conns=conns, anomaly=anomaly)
NORMAL = tele(15.0, 18, 8)

CORPUS = [
 dict(id="R1", device="Tenda W18E", version="16.01.0.11",
      issue="Buffer Overflow in goform/setModules via Password argument.",
      solution="Block TCP port 8080 upstream. Disable remote web management. Update to firmware v16.01.0.12.",
      tokens=["8080", "16.01.0.12"], allowed=["16.01.0.11", "16.01.0.12"], shell=False,
      tele=tele(99.4, 4820, 96),
      q_gu="My Tenda router is very slow and its admin page is being hit from the internet. What should I do?"),
 dict(id="R2", device="SmartCam Pro", version="",
      issue="Hardcoded default credentials in telnetd and exposed RTSP stream.",
      solution="Block ports 23 and 554 at firewall. Force password reset via local SSH.",
      tokens=["23", "554"], allowed=[], shell=False,
      tele=tele(88.5, 940, 93),
      q_gu="My SmartCam indoor camera light blinks fast and my internet is slow. What is going on?"),
 dict(id="R3", device="Industrial Sensor X", version="",
      issue="Unencrypted MQTT telemetry data transmission on port 1883 allowing MITM packet spoofing.",
      solution="Disable port 1883 and implement mTLS on port 8883 for MQTT brokers.",
      tokens=["1883", "8883"], allowed=[], shell=True,
      tele=NORMAL,
      q_gu="The Industrial Sensor X in our workshop shows a warning about unencrypted data. Is that dangerous?"),
 dict(id="R4", device="D-Link DIR-859", version="",
      issue="Command injection in /hedwig.cgi via HTTP Cookie.",
      solution="Disable UPnP. Filter UDP 1900 on WAN. Patch to v1.06b01.",
      tokens=["1900", "1.06b01"], allowed=["1.06b01"], shell=False,
      tele=NORMAL,
      q_gu="Is it safe to leave UPnP switched on in my D-Link router?"),
 dict(id="R5", device="MatterSmart Hub", version="",
      issue="Null-pointer dereference during PASE handshake.",
      solution="Limit BLE commissioning to 180s. Rate-limit PASE. OTA to v1.2.1.",
      tokens=["180", "1.2.1"], allowed=["1.2.1"], shell=False,
      tele=NORMAL,
      q_gu="My MatterSmart hub sometimes restarts when I add a new device. Is that a security problem?"),
 dict(id="R6", device="Hikvision Camera", version="",
      issue="Improper auth check in /Security/users leaks cleartext passwords.",
      solution="Isolate on CCTV VLAN. Block WAN port 80. Flash v5.4.5.",
      tokens=["80", "5.4.5"], allowed=["5.4.5"], shell=False,
      tele=tele(64.0, 310, 91),
      q_gu="Someone may have downloaded the settings file from my Hikvision camera. What do I do?"),
 dict(id="R7", device="Siemens S7-1200", version="",
      issue="Missing replay protection in S7comm tokens.",
      solution="Enable S7comm-Plus TLS integrity. Patch v4.5.2.",
      tokens=["4.5.2"], allowed=["4.5.2"], shell=False,
      tele=NORMAL,
      q_gu="Our Siemens S7-1200 controller stopped once by itself. Could someone have sent it a command?"),
 dict(id="R8", device="Netgear R7000P", version="",
      issue="Pre-auth stack buffer overflow in upnpd SOAP request.",
      solution="Disable UPnP service. Flash v1.3.3.140.",
      tokens=["1.3.3.140"], allowed=["1.3.3.140"], shell=False,
      tele=tele(97.1, 3900, 94),
      q_gu="My Netgear router became unresponsive and the status light is red. Could it be hacked?"),
]
REC = {r["id"]: r for r in CORPUS}
with open("iot_threats.json", "w") as f:
    json.dump(CORPUS, f, indent=2)

OUT_OF_CORPUS = [
 ("General User", "My smart thermostat keeps rebooting. How can I secure it?"),
 ("Firmware Developer", "How should I protect a Zigbee door lock from replay attacks?"),
 ("Network Administrator", "Our LoRaWAN gateway shows duplicated uplink frames. Is this an attack and what should we do?"),
 ("General User", "Is it safe to let my smart TV connect to the guest Wi-Fi?"),
 ("Firmware Developer", "How do I add secure boot to an ESP32 sensor node?"),
 ("Network Administrator", "How should we segment smart lighting controllers from the office LAN?"),
]

CASES = []
for r in CORPUS:
    explicit = f"What is the vulnerability with the {r['device']} and how do I fix it?"
    for role, q, style in [(ROLES[0], r["q_gu"], "symptom"),
                           (ROLES[1], explicit, "explicit"),
                           (ROLES[2], explicit, "explicit")]:
        CASES.append(dict(case_id=f"{r['id']}-{role.split()[0][:3]}", rec=r["id"], role=role,
                          query=q, tele=r["tele"], style=style))
for i, (role, q) in enumerate(OUT_OF_CORPUS):
    CASES.append(dict(case_id=f"OOC-{i+1}", rec=None, role=role, query=q, tele=NORMAL, style="ooc"))
CASE = {c["case_id"]: c for c in CASES}

emb = SentenceTransformer("all-MiniLM-L6-v2")
enc = lambda t: emb.encode(t, normalize_embeddings=True)
chroma = chromadb.Client()
col = chroma.get_or_create_collection(name="iot_kb")
DOC_EMB = {r["id"]: enc(f"Device: {r['device']} | Issue: {r['issue']}") for r in CORPUS}
if col.count() == 0:
    for r in CORPUS:
        col.add(ids=[r["id"]], embeddings=[DOC_EMB[r["id"]].tolist()],
                documents=[f"{r['device']} | {r['issue']} | Fix: {r['solution']}"])

def retrieve(query, k=3):
    q = enc(query)
    ids = col.query(query_embeddings=[q.tolist()], n_results=k)["ids"][0]
    return ids, [float(np.dot(q, DOC_EMB[i])) for i in ids]

def record_str(rec):
    if rec is None: return "NO VERIFIED RECORD FOUND for this query."
    return f"{rec['device']} | Issue: {rec['issue']} | Verified fix: {rec['solution']}"

def fuse(t):
    tag = "[CRITICAL ACTIVE ATTACK] " if (t["anomaly"] >= 80 or t["cpu"] >= 90) else "[NORMAL STATE] "
    return f"{tag}CPU_Load: {t['cpu']}% | Active_Conns: {t['conns']} | Anomaly_Score: {t['anomaly']}/100"

def p_llm_only(role, q):
    return f"Role: {role}. Query: {q}. Formulate a fix."

def p_icot1(V, role, q):
    return (f"Vulnerability description: {V}\nUser role: {role}\nUser question: {q}\n"
            'Analyze the user and the vulnerability. Output JSON only with keys "role" '
            '{"knowledge","goals","requirements"} and "vulnerability" {"type","location","impact"}.')

def p_icot2(V, Sg, role, q):
    return (f"{V}\n{Sg}\nUse the relevant information above to answer the following questions:\n"
            f"Role: {role}\nQuestion: {q}")

def p_grounded(role, q, record, tele_block):
    tl = f"Live device telemetry: {tele_block}\n" if tele_block else ""
    tr = ("- If telemetry shows an active attack, list containment steps first and firmware updates "
          "only after the device is contained.\n") if tele_block else ""
    return (f"Role: {role}\nQuery: {q}\n{tl}Verified knowledge-base record: {record}\n\nRules:\n"
            "- Device-specific facts (ports, firmware versions, commands) must come only from the verified "
            "record. Never invent firmware versions or shell commands.\n"
            "- If the record is missing or describes a different device than the user's, say so and give "
            f"only general, version-free advice.\n{tr}"
            "Think step by step silently (role, record and telemetry, impact, safest ordered fix), then "
            "output only the final answer as short numbered steps suited to the role. "
            "State the vulnerability once; do not repeat it.")

def p_critique(draft, record, tele_block):
    t = f"Telemetry: {tele_block}\n" if tele_block else ""
    return (f"Draft answer:\n{draft}\n\nVerified record: {record}\n{t}\n"
            "Check the draft. It FAILS if it contains any port, firmware version or command that is not in "
            "the verified record, or (when telemetry shows an active attack) if a firmware update comes "
            "before containment. Generic advice with no version, port or command is fine.\n"
            "If it passes, reply with exactly: OK\n"
            "If it fails, reply with: REWRITE: followed by the corrected answer, using only facts from the record.")

def run_c1(c):
    return dict(response=gen(p_llm_only(c["role"], c["query"])))

def run_c2(c):
    V = REC[c["rec"]]["issue"] if c["rec"] else "(no vulnerability description provided)"
    Sg = gen(p_icot1(V, c["role"], c["query"]))
    return dict(response=gen(p_icot2(V, Sg, c["role"], c["query"])))

def ram(c, use_tele, use_critique):
    ids, sims = retrieve(c["query"])
    rec = REC[ids[0]] if sims[0] >= SIM_MIN else None
    record = record_str(rec)
    tb = fuse(c["tele"]) if use_tele else ""
    draft = gen(p_grounded(c["role"], c["query"], record, tb))
    out = dict(retrieved=ids[0], sim=sims[0], abstained=rec is None, draft=draft, response=draft)
    if use_critique:
        crit = gen(p_critique(draft, record, tb))
        up = crit.strip().upper()
        out["critique"] = crit
        out["parse_ok"] = up.startswith("OK") or up.startswith("REWRITE")
        out["refined"] = up.startswith("REWRITE") and ":" in crit
        if out["refined"]:
            rewritten = crit.split(":", 1)[1].strip()
            if rewritten:
                out["response"] = rewritten
    return out

CONDS = {"C1": ("LLM-only", run_c1),
         "C2": ("ICoT (our reimpl.)", run_c2),
         "C3": ("RAG only", lambda c: ram(c, False, False)),
         "C4": ("RAG + telemetry", lambda c: ram(c, True, False)),
         "C5": ("Full RAM-CoT", lambda c: ram(c, True, True))}

VER_RE  = re.compile(r"(?<![\w.])v?(\d+\.\d+\.\d+(?:\.\d+)?|\d+\.\d+[a-z]\d+)(?![\w])", re.I)
CMD_RE  = re.compile(r"\b(sudo|apt-get|apt install|apt update|yum|dnf|systemctl|ufw|firewall-cmd)\b", re.I)
CONTAIN = re.compile(r"\b(block|drop|isolate|unplug|disconnect|quarantine|deny|cut off|take (?:it )?offline|power off|shut ?down)\b", re.I)
FLASH   = re.compile(r"\b(flash(?:ing)?|firmware update|update (?:the |to )?(?:firmware|v?\d)|upgrade|install (?:the )?(?:patch|firmware|update)|apply (?:the )?(?:patch|update))\b", re.I)
PRIVATE = ("192.168.", "10.", "172.16.", "172.17.", "172.31.", "169.254.")
def norm_ver(v): return v.lower().lstrip("v")
def version_tokens(text):
    out = []
    for m in VER_RE.finditer(text):
        tok = norm_ver(m.group(1))
        if not tok.startswith(PRIVATE):
            out.append(tok)
    return out
def has_token(text, tok):
    return re.search(r"(?<![\d.])" + re.escape(tok) + r"(?!\d)", text) is not None
def score_response(case, text):
    rec = REC.get(case["rec"]) if case["rec"] else None
    allowed = {norm_ver(v) for v in rec["allowed"]} if rec else set()
    m = {}
    m["fr"] = (sum(has_token(text, t) for t in rec["tokens"]) / len(rec["tokens"])) if rec else np.nan
    m["uv"] = float(any(v not in allowed for v in version_tokens(text)))
    m["uc"] = float(bool(CMD_RE.search(text))) if (rec is None or not rec["shell"]) else 0.0
    is_attack = rec is not None and (case["tele"]["anomaly"] >= 80 or case["tele"]["cpu"] >= 90)
    if is_attack:
        c = CONTAIN.search(text); f = FLASH.search(text)
        m["cf"] = float(bool(c) and (f is None or c.start() < f.start()))
    else:
        m["cf"] = np.nan
    words = re.findall(r"\w+", text.lower())
    m["words"] = len(words)
    tri = list(zip(words, words[1:], words[2:]))
    m["rep3"] = (1 - len(set(tri)) / len(tri)) if tri else 0.0
    return m

# ===== NEW: no-guardrail variant + C6/C7 conditions =====
def p_grounded_no_guardrail(role, q, record, tele_block):
    tl = f"Live device telemetry: {tele_block}\n" if tele_block else ""
    tr = ("- If telemetry shows an active attack, list containment steps first and firmware updates "
          "only after the device is contained.\n") if tele_block else ""
    return (f"Role: {role}\nQuery: {q}\n{tl}Verified knowledge-base record: {record}\n\n"
            f"{tr}"
            "Think step by step silently (role, record and telemetry, impact, safest ordered fix), then "
            "output only the final answer as short numbered steps suited to the role. "
            "State the vulnerability once; do not repeat it.")

def ram_variant(c, use_tele, use_critique, guarded=True):
    ids, sims = retrieve(c["query"])
    rec = REC[ids[0]] if sims[0] >= SIM_MIN else None
    record = record_str(rec)
    tb = fuse(c["tele"]) if use_tele else ""
    prompt_fn = p_grounded if guarded else p_grounded_no_guardrail
    draft = gen(prompt_fn(c["role"], c["query"], record, tb))
    out = dict(retrieved=ids[0], sim=sims[0], abstained=rec is None, draft=draft, response=draft)
    if use_critique:
        crit = gen(p_critique(draft, record, tb))
        up = crit.strip().upper()
        out["critique"] = crit
        out["parse_ok"] = up.startswith("OK") or up.startswith("REWRITE")
        out["refined"] = up.startswith("REWRITE") and ":" in crit
        if out["refined"]:
            rewritten = crit.split(":", 1)[1].strip()
            if rewritten:
                out["response"] = rewritten
    return out

CONDS["C6"] = ("RAG+tele, no guardrail", lambda c: ram_variant(c, True, False, guarded=False))
CONDS["C7"] = ("RAG+tele+critique, no guardrail", lambda c: ram_variant(c, True, True, guarded=False))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [30]:
# %% Cell 3 - Retrieval-only Check
rows = []
for c in CASES:
    ids, sims = retrieve(c["query"])
    rows.append(dict(case=c["case_id"], style=c["style"], truth=c["rec"], top1=ids[0], sim=round(sims[0], 3)))
rdf = pd.DataFrame(rows)
print(rdf.to_string(index=False))
inc = rdf[rdf.truth.notna()]
print("\ntop-1 accuracy:", round((inc.top1 == inc.truth).mean(), 3))

  case    style truth top1   sim
R1-Gen  symptom    R1   R1 0.270
R1-Fir explicit    R1   R1 0.512
R1-Net explicit    R1   R1 0.512
R2-Gen  symptom    R2   R2 0.406
R2-Fir explicit    R2   R2 0.636
R2-Net explicit    R2   R2 0.636
R3-Gen  symptom    R3   R3 0.454
R3-Fir explicit    R3   R3 0.534
R3-Net explicit    R3   R3 0.534
R4-Gen  symptom    R4   R8 0.304
R4-Fir explicit    R4   R4 0.433
R4-Net explicit    R4   R4 0.433
R5-Gen  symptom    R5   R5 0.531
R5-Fir explicit    R5   R5 0.625
R5-Net explicit    R5   R5 0.625
R6-Gen  symptom    R6   R6 0.541
R6-Fir explicit    R6   R6 0.626
R6-Net explicit    R6   R6 0.626
R7-Gen  symptom    R7   R7 0.534
R7-Fir explicit    R7   R7 0.580
R7-Net explicit    R7   R7 0.580
R8-Gen  symptom    R8   R8 0.227
R8-Fir explicit    R8   R8 0.448
R8-Net explicit    R8   R8 0.448
 OOC-1      ooc  None   R3 0.227
 OOC-2      ooc  None   R7 0.374
 OOC-3      ooc  None   R6 0.289
 OOC-4      ooc  None   R2 0.296
 OOC-5      ooc  None   R3 0.284
 OOC-6    

In [40]:
# %% Cell 4 - Main Experiment Loop
CACHE = "raw_results.jsonl"
done = {}
if os.path.exists(CACHE):
    for line in open(CACHE):
        d = json.loads(line); done[d["key"]] = d
with open(CACHE, "a") as fh:
    for c in CASES:
        for cid, (name, fn) in CONDS.items():
            key = f"{c['case_id']}|{cid}"
            if key in done:
                continue
            t0 = time.time()
            out = fn(c)
            out["latency"] = time.time() - t0
            out.update(key=key, case_id=c["case_id"], cond=cid)
            fh.write(json.dumps(out, default=str) + "\n"); fh.flush()
            done[key] = out
            print("done", key, round(out["latency"], 1), "s")

print("API calls this session:", N_CALLS, "| empty outputs:", N_EMPTY)

done R4-Gen|C6 5.2 s
  [retry] 0 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high deman
  [retry] 1 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high deman
done R4-Gen|C7 60.2 s
  [retry] 0 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high deman
done R4-Fir|C6 20.7 s
done R4-Fir|C7 10.2 s
done R4-Net|C6 5.3 s
done R4-Net|C7 10.8 s
done R5-Gen|C6 10.6 s
  [retry] 0 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high deman
done R5-Gen|C7 26.0 s
done R5-Fir|C6 5.1 s
done R5-Fir|C7 13.6 s
done R5-Net|C6 5.1 s
done R5-Net|C7 14.3 s
done R6-Gen|C6 7.4 s
done R6-Gen|C7 10.7 s
done R6-Fir|C6 5.2 s
done R6-Fir|C7 10.3 s
done R6-Net|C6 5.5 s
  [retry] 0 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high deman
done R6-Net|C7 26.9 s
done R7-Gen|C6 5.4 s
done R7-Gen|C7 11.5 s
don

In [44]:
import json

SIM_MIN = 0.30  # match your notebook's value

fixed = []
with open("raw_results.jsonl") as f:
    for line in f:
        d = json.loads(line)
        if "sim" in d:
            d["abstained"] = d["sim"] < SIM_MIN
        fixed.append(d)

with open("raw_results.jsonl", "w") as f:
    for d in fixed:
        f.write(json.dumps(d, default=str) + "\n")

print("Fixed", sum(1 for d in fixed if "sim" in d), "RAG-condition rows")

Fixed 150 RAG-condition rows


In [46]:
# %% Cell 5 - Score everything, write LaTeX tables and figure
recs = []
for d in done.values():
    c = CASE[d["case_id"]]
    m = score_response(c, d.get("response", ""))
    m.update(case_id=d["case_id"], cond=d["cond"], role=c["role"], latency=d["latency"],
             empty=int(not d.get("response", "").strip()))
    recs.append(m)
df = pd.DataFrame(recs)
df.to_csv("scored_results.csv", index=False)
print("empty responses:", int(df['empty'].sum()))

names = {k: v[0] for k, v in CONDS.items()}
pct = lambda x: f"{100*x:.1f}"

L = [r"\begin{tabular}{@{}lrrrrrrr@{}}", r"\toprule",
     r"Condition & FR$\uparrow$ & UV$\downarrow$ & UC$\downarrow$ & CF$\uparrow$ & Words & Rep3 & Lat.\ (s) \\", r"\midrule"]
for k in CONDS:
    s = df[df.cond == k]
    L.append(f"{names[k]} & {pct(s.fr.mean())} & {pct(s.uv.mean())} & {pct(s.uc.mean())} & {pct(s.cf.mean())} & "
             f"{s.words.mean():.0f} & {pct(s.rep3.mean())} & {s.latency.mean():.1f}$\\pm${s.latency.std():.1f} \\\\")
L += [r"\bottomrule", r"\end{tabular}"]
open("tab_main.tex", "w").write("\n".join(L))

L = [r"\begin{tabular}{@{}lrrrrrr@{}}", r"\toprule",
     r" & \multicolumn{3}{c}{FR (\%)} & \multicolumn{3}{c}{Words} \\ \cmidrule(lr){2-4}\cmidrule(l){5-7}",
     r"Condition & GU & Dev & NA & GU & Dev & NA \\", r"\midrule"]
for k in CONDS:
    cells = []
    for metric, f in [("fr", pct), ("words", lambda x: f"{x:.0f}")]:
        for r in ROLES:
            s = df[(df.cond == k) & (df.role == r)][metric]
            cells.append(f(s.mean()) if s.notna().any() else "--")
    L.append(f"{names[k]} & " + " & ".join(cells) + r" \\")
L += [r"\bottomrule", r"\end{tabular}"]
open("tab_role.tex", "w").write("\n".join(L))

inc = rdf[rdf.truth.notna()]; ooc = rdf[rdf.truth.isna()]
acc = lambda s: pct((s.top1 == s.truth).mean())
L = [r"\begin{tabular}{@{}lr@{}}", r"\toprule", r"Measure & Value \\", r"\midrule",
     f"Top-1 accuracy, symptom queries (n=8) & {acc(inc[inc['style']=='symptom'])}\\% \\\\",
     f"Top-1 accuracy, explicit queries (n=16) & {acc(inc[inc['style']=='explicit'])}\\% \\\\",
     f"Mean similarity, correct top-1 hits & {inc[inc.top1==inc.truth].sim.mean():.2f} \\\\",
     f"Mean similarity, out-of-corpus queries (n={len(ooc)}) & {ooc.sim.mean():.2f} \\\\",
     f"In-corpus queries rejected by gate ($\\delta$={SIM_MIN}) & {(inc.sim < SIM_MIN).sum()} / {len(inc)} \\\\",
     f"Out-of-corpus queries rejected by gate & {(ooc.sim < SIM_MIN).sum()} / {len(ooc)} \\\\",
     r"\bottomrule", r"\end{tabular}"]
open("tab_retr.tex", "w").write("\n".join(L))

c5 = [d for d in done.values() if d["cond"] == "C5"]
n = len(c5); refined = sum(bool(d.get("refined")) for d in c5)
pf = sum(not d.get("parse_ok", True) for d in c5)
L = [r"\begin{tabular}{@{}lr@{}}", r"\toprule", r"Critique-node statistic & Value \\", r"\midrule",
     f"Drafts audited & {n} \\\\",
     f"Approved on first pass (OK) & {n-refined-pf} \\\\",
     f"Rewritten by the critique & {refined} ({pct(refined/n)}\\%) \\\\",
     f"Reply in neither OK nor REWRITE format & {pf} \\\\",
     f"Mean latency, RAG+telemetry vs.\\ full (s) & {df[df.cond=='C4'].latency.mean():.1f} vs.\\ {df[df.cond=='C5'].latency.mean():.1f} \\\\",
     r"\bottomrule", r"\end{tabular}"]
open("tab_crit.tex", "w").write("\n".join(L))

import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
mets = [("fr", "Fact recall (higher better)"), ("uv", "Unverified version (lower better)"),
        ("uc", "Unsupported command (lower better)"), ("cf", "Containment first (higher better)")]
fig, axs = plt.subplots(1, 4, figsize=(11, 2.6))
for ax, (m, title) in zip(axs, mets):
    ax.bar(list(CONDS.keys()), [100 * df[df.cond == k][m].mean() for k in CONDS])
    ax.set_title(title, fontsize=8); ax.set_ylim(0, 100); ax.tick_params(labelsize=7)
axs[0].set_ylabel("%", fontsize=8)
plt.tight_layout(); plt.savefig("fig_results.png", dpi=200)
print(open("tab_main.tex").read())

empty responses: 2
\begin{tabular}{@{}lrrrrrrr@{}}
\toprule
Condition & FR$\uparrow$ & UV$\downarrow$ & UC$\downarrow$ & CF$\uparrow$ & Words & Rep3 & Lat.\ (s) \\
\midrule
LLM-only & 12.5 & 6.7 & 0.0 & 33.3 & 433 & 1.5 & 13.7$\pm$11.2 \\
ICoT (our reimpl.) & 16.7 & 0.0 & 0.0 & 25.0 & 397 & 1.4 & 22.4$\pm$18.0 \\
RAG only & 83.3 & 0.0 & 0.0 & 58.3 & 77 & 0.4 & 14.9$\pm$30.5 \\
RAG + telemetry & 81.2 & 0.0 & 0.0 & 91.7 & 77 & 0.1 & 9.2$\pm$6.0 \\
Full RAM-CoT & 70.8 & 6.7 & 0.0 & 83.3 & 60 & 0.1 & 40.0$\pm$114.1 \\
RAG+tele, no guardrail & 87.5 & 0.0 & 0.0 & 100.0 & 64 & 0.3 & 7.3$\pm$4.5 \\
RAG+tele+critique, no guardrail & 75.0 & 6.7 & 0.0 & 100.0 & 54 & 0.1 & 16.5$\pm$10.9 \\
\bottomrule
\end{tabular}


In [48]:
# %% Cell 6 - Demo Run for Screenshots
demo = CASE["R1-Net"]
out = ram(demo, True, True)
lines = [f"Role: {demo['role']}", f"Query: {demo['query']}", "",
         f"Retrieved: {out['retrieved']} (cosine {out['sim']:.2f})", f"Telemetry: {fuse(demo['tele'])}", "",
         "--- DRAFT ---", out["draft"], "", f"--- CRITIQUE (refined={out['refined']}) ---", out["critique"],
         "", "--- FINAL ---", out["response"]]
text = "\n".join(lines).encode("ascii", "replace").decode()
wrapped = "\n".join(textwrap.fill(l, 62) if l.strip() else "" for l in text.split("\n"))
open("demo_output.txt", "w").write(wrapped)
print(wrapped)

Role: Network Administrator
Query: What is the vulnerability with the Tenda W18E and how
do I fix it?

Retrieved: R1 (cosine 0.51)
Telemetry: [CRITICAL ACTIVE ATTACK] CPU_Load: 99.4% |
Active_Conns: 4820 | Anomaly_Score: 96/100

--- DRAFT ---
1. The Tenda W18E has a buffer overflow vulnerability in
`goform/setModules` via the `Password` argument.
2. Block TCP port 8080 upstream immediately to stop the active
attack.
3. Disable remote web management.
4. Update to firmware v16.01.0.12.

--- CRITIQUE (refined=True) ---
REWRITE:
1. The Tenda W18E has a buffer overflow vulnerability in
`goform/setModules` via the `Password` argument.
2. Block TCP port 8080 upstream immediately to stop the active
attack.
3. Disable remote web management.
4. Update to firmware v16.01.0.12.

--- FINAL ---
1. The Tenda W18E has a buffer overflow vulnerability in
`goform/setModules` via the `Password` argument.
2. Block TCP port 8080 upstream immediately to stop the active
attack.
3. Disable remote web managemen